# 05x_feature_contract_rebuild_260515

91개 전체 컬럼 재검토 및 사용자 승인용 feature contract 작성.
모델링, EDA, SHAP, Optuna, segmentation은 이 단계에서 수행하지 않는다.

In [1]:
import pandas as pd
import zipfile
from pathlib import Path
from datetime import datetime

PARK = Path('C:/Code/ott-churn-prediction/park.ingyeom')
SOURCE_CSV = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
OUT_BASE = PARK / 'reports' / 'audits' / '05x_feature_contract_rebuild_260515'
NB_DIR = PARK / 'notebook' / '05x_feature_contract_rebuild_260515'
ZIP_OUT = PARK / 'zip' / '05x_feature_contract_rebuild_260515_review_package.zip'
NOTE_MD = PARK / 'note.md'

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_DIR = OUT_BASE / f'run_{ts}' if OUT_BASE.exists() else OUT_BASE
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output dir: {OUT_DIR}')
print(f'Timestamp: {ts}')

Output dir: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\05x_feature_contract_rebuild_260515
Timestamp: 20260515_023135


In [2]:
# PREFLIGHT
stop_reasons = []

src_exists = SOURCE_CSV.exists()
if not src_exists:
    stop_reasons.append('source_master_csv_missing')

df_full = pd.read_csv(SOURCE_CSV) if src_exists else None
row_count = len(df_full) if df_full is not None else None
col_count = len(df_full.columns) if df_full is not None else None

active_steps = [
    '01_data_contract_260513', '02_target_score_orientation_260513',
    '03_observation_window_policy_260513', '04_promotion_split_260513'
]
active_ok = all((PARK / 'notebook' / s).exists() for s in active_steps)
archive_exists = (PARK / '_archive' / 'pre13b_conservative_safe_22_reference').exists()

preflight = pd.DataFrame([
    {'check': 'source_master_exists', 'status': 'PASS' if src_exists else 'FAIL', 'detail': str(SOURCE_CSV)},
    {'check': 'row_count', 'status': 'PASS' if row_count == 23343 else f'WARN:{row_count}', 'detail': str(row_count)},
    {'check': 'column_count', 'status': 'PASS' if col_count == 91 else 'FAIL', 'detail': str(col_count)},
    {'check': 'active_steps_01_04_exist', 'status': 'PASS' if active_ok else 'FAIL', 'detail': str(active_ok)},
    {'check': 'archive_reference_exists', 'status': 'PASS' if archive_exists else 'WARN', 'detail': str(archive_exists)},
    {'check': 'output_dir_created', 'status': 'PASS', 'detail': str(OUT_DIR)},
    {'check': 'stop_reason', 'status': 'PASS' if not stop_reasons else 'FAIL', 'detail': '; '.join(stop_reasons) if stop_reasons else 'none'},
])
preflight.to_csv(OUT_DIR / '05x_preflight_input_validation.csv', index=False)
print(preflight.to_string(index=False))
if stop_reasons:
    raise RuntimeError(f'Preflight failed: {stop_reasons}')

                   check status                                                                                       detail
    source_master_exists   PASS   C:\Code\ott-churn-prediction\park.ingyeom\data\(광일)Membership_v2_with_derived_features.csv
               row_count   PASS                                                                                        23343
            column_count   PASS                                                                                           91
active_steps_01_04_exist   PASS                                                                                         True
archive_reference_exists   WARN                                                                                        False
      output_dir_created   PASS C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\05x_feature_contract_rebuild_260515
             stop_reason   PASS                                                                                         none


In [3]:
# FULL COLUMN INVENTORY
inventory_rows = []
for col in df_full.columns:
    s = df_full[col]
    s_drop = s.dropna()
    inventory_rows.append({
        'column_order': int(df_full.columns.get_loc(col)),
        'column_name': col,
        'dtype': str(s.dtype),
        'missing_count': int(s.isna().sum()),
        'unique_count': int(s.nunique()),
        'sample_min': str(s_drop.min()) if len(s_drop) > 0 else 'n/a',
        'sample_max': str(s_drop.max()) if len(s_drop) > 0 else 'n/a',
    })
inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(OUT_DIR / '05x_full_column_inventory.csv', index=False)
print(f'Column inventory: {len(inventory_df)} rows')
print(inventory_df[['column_order','column_name','dtype','missing_count','unique_count']].to_string(index=False))

Column inventory: 91 rows
 column_order                column_name   dtype  missing_count  unique_count
            0                   USER_KEY     str              0         23134
            1               product_code     str              0            46
            2                      price float64              0            32
            3             billing_method   int64              0             9
            4                 max_screen float64              0             3
            5               is_promotion   int64              0             2
            6         is_churn_prevented   int64              0             2
            7             payment_device     str              0             6
            8           is_user_verified   int64              0             2
            9                     gender     str              0             3
           10                        age float64              0            12
           11                   reg_da

In [4]:
# DECISION TABLE CONSTRUCTION

CONSERVATIVE_22 = {
    'avg_gap_w1_watch_days','avg_gap_w2_watch_days','avg_gap_w3_watch_days',
    'is_cold_start_3d','is_cold_start_7d',
    'watch_time(min)_w1','watch_time(min)_w2','watch_time(min)_w3',
    'watch_session_w1','watch_session_w2','watch_session_w3',
    'retention_w2_ratio','retention_w3_ratio',
    'diff_between_w2_w1','diff_between_w3_w1','diff_between_w3_w2',
    'is_w1_over_50pct','is_w2_over_50pct','is_w3_over_50pct',
    'is_only_w1','is_only_w2','is_only_w3',
}

EXPLICIT = {
    'USER_KEY': ('identifier','id','row_id','row identifier','yes','n/a','n/a','n/a','n/a','forbidden_or_audit_only',
        'Row identifier; never a model feature; use as group key in CV only','CV group key only','no'),
    'is_repurchase': ('target','target','target_label','next-month repurchase label','yes','post-scoring','high','high','target_itself',
        'forbidden_or_audit_only','Target variable; including as feature causes direct leakage','Never use as feature','no'),
    'end_date': ('subscription_metadata','subscription_end','event_timestamp','subscription end date','yes','unclear','high','high','medium',
        'high','forbidden_or_audit_only',
        'May encode outcome; unclear if observable at day21 scoring point','Confirm day21 observability before any use','yes'),
    'is_churn_prevented': ('intervention_outcome','churn_defense','outcome_flag','churn defense intervention flag','yes','unclear','high','medium','medium',
        'high','forbidden_or_audit_only',
        'If current-cycle outcome: leakage. If historical: potentially safe. Confirm timing.','Must confirm historical vs current-cycle','yes'),
    'is_promotion': ('split_variable','top_level_split','split_axis','top-level promotion split','yes','yes','low','low','low',
        'low','forbidden_or_audit_only',
        'Split axis only; allowed only in overall_with_promotion scope; forbidden in groupwise models','Exclude from promotion_only/nonpromotion_only','no'),
    'reg_date': ('registration_metadata','timestamp','day0_anchor','day0 registration anchor','yes','yes','medium','low','low',
        'low','forbidden_or_audit_only',
        'Raw date causes cohort collinearity; prefer derived features (reg_hour, reg_is_weekend)','Do not use raw reg_date as feature','yes'),
    'recency': ('usage_recency','recency','usage_recency','days since last viewing','yes','unclear','high','medium','medium',
        'medium','unresolved_user_review_required',
        'Reference date unknown; must confirm it is day21 or earlier as reference point','If reference_date > day21 this leaks future behavior','yes'),
}

MEMBERSHIP_COLS = {
    'product_code','price','billing_method','max_screen','payment_device',
    'is_user_verified','gender','age','is_standard','is_premium',
    'age_group','is_female','is_male','reg_hour','reg_is_weekend',
    'reg_hour_morning','reg_hour_afternoon','reg_hour_evening','reg_hour_night',
    'payment_is_mobile','payment_is_pc','payment_is_android','payment_is_ios',
}

TOTAL_USAGE_COLS = {
    'total_watch_count','total_watch_time(min)','watch_days','active_ratio',
    'watch_per_day','avg_watch_time(min)','median_watch_time(min)','std_watch_time(min)',
    'avg_daily_watch_time(min)','max_watch_time(min)','max_daily_watch_time(min)',
    'max_daily_sessions','unique_movie','avg_gap_between_watch_days','max_inactive_gap_days',
    'avg_rewatch_ratio','weekend_watch_ratio','watch_ratio_under_1m','watch_ratio_under_5m',
    'movie_per_active_day','max_day_share','day_count_over_3times',
}

GENRE_CONTENT_COLS = {
    'action_adventure_ratio','family_animation_ratio','drama_ratio','thriller_crime_ratio',
    'sf_fantasy_ratio','comedy_ratio','romance_ratio','horror_ratio','documentary_ratio',
    'historical_war_ratio','other_ratio','genre_diversity_count','old_movie_ratio(5y)',
    'avg_ott_release_year','new_movie_in_90d_ratio','new_movie_in_180d_ratio','new_movie_in_365d_ratio',
}

def classify(col):
    if col in CONSERVATIVE_22:
        return ('weekly_behavior','conservative_window','weekly_feature',
            'day0-20 weekly viewing behavior','yes','yes','low','low','low','low',
            'keep_in_conservative_safe_22',
            'Confirmed weekly-window feature; validated in pre-13b baseline','','no')
    if col in EXPLICIT:
        e = EXPLICIT[col]
        # handle different tuple lengths
        if len(e) == 13:
            return e
        return e
    if col in MEMBERSHIP_COLS:
        return ('membership_context','membership_or_demographic','membership_or_registration',
            'user/plan characteristics known at registration','yes','yes','low','low','low','low',
            'candidate_for_expanded_feature_set',
            'Known at subscription start; available at day21 scoring point',
            'Check redundancy between derived and parent columns','yes')
    if col in TOTAL_USAGE_COLS:
        return ('usage_behavior_summary','total_or_aggregate','usage_aggregate',
            'aggregate viewing behavior metric','yes','likely_yes','medium','low','low','low',
            'candidate_for_expanded_with_caveat',
            '09b archive validated total_watch_time/count match day0-20; other aggregate cols likely same window but naming suggests all-period',
            'total_ naming misleading; confirm all cols are day0-20 window','yes')
    if col in GENRE_CONTENT_COLS:
        return ('content_preference','genre_or_content','content_feature',
            'content/genre preference during observation window','yes','likely_yes','medium','low','low','low',
            'candidate_for_expanded_with_caveat',
            'Likely computed from day0-20 window; 09b found 206-row genre mismatch due to Movie_Master MOVIE_NUM duplication caveat',
            'Confirm observation window is day0-20; genre mismatch caveat from 09b still open','yes')
    return ('unknown','unknown','unknown','unknown','unknown','unknown','unknown','unknown','unknown','unknown',
        'unresolved_user_review_required','Pattern not matched; manual review required','Review manually','yes')

KEYS = ['feature_family','sub_family','original_role_if_known','likely_business_intent',
        'team_feature_or_source_column','availability_at_day21',
        'timing_risk','leakage_risk','target_proxy_risk','response_period_risk',
        'llm_proposed_decision','reason','caution','user_approval_required']

decision_rows = []
for _, inv_row in inventory_df.iterrows():
    col = inv_row['column_name']
    vals = classify(col)
    row = {'column_name': col, 'dtype': inv_row['dtype'],
           'missing_count': inv_row['missing_count'], 'unique_count': inv_row['unique_count']}
    # handle explicit with different schema
    if col in EXPLICIT and len(EXPLICIT[col]) == 13:
        e = EXPLICIT[col]
        row.update(dict(zip(
            ['feature_family','sub_family','original_role_if_known','likely_business_intent',
             'team_feature_or_source_column','availability_at_day21',
             'timing_risk','leakage_risk','target_proxy_risk','response_period_risk',
             'llm_proposed_decision','reason','caution','user_approval_required'],
            list(e[:6]) + list(e[6:])
        )))
    else:
        row.update(dict(zip(KEYS, vals)))
    row['evidence_source'] = '05x_rule_based_classification'
    row['final_decision_status'] = 'pending_user_approval'
    decision_rows.append(row)

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUT_DIR / '05x_feature_resolution_decision_table.csv', index=False)
print(f'Decision table: {len(decision_df)} rows')
print(decision_df['llm_proposed_decision'].value_counts())

Decision table: 91 rows
llm_proposed_decision
candidate_for_expanded_with_caveat                                    39
candidate_for_expanded_feature_set                                    23
keep_in_conservative_safe_22                                          22
forbidden_or_audit_only                                                4
Row identifier; never a model feature; use as group key in CV only     1
Target variable; including as feature causes direct leakage            1
unresolved_user_review_required                                        1
Name: count, dtype: int64


In [5]:
# CONTRACT FILES

# 1. conservative_safe_22
c22 = decision_df[decision_df['llm_proposed_decision']=='keep_in_conservative_safe_22'].copy()
c22['plan'] = 'conservative_safe_22'
c22.to_csv(OUT_DIR / '05x_conservative_safe_22_contract.csv', index=False)
print(f'conservative_safe_22: {len(c22)} features')

# 2. expanded candidate
expanded_decisions = {'candidate_for_expanded_feature_set','candidate_for_expanded_with_caveat','keep_in_conservative_safe_22'}
exp = decision_df[decision_df['llm_proposed_decision'].isin(expanded_decisions)].copy()
exp['in_conservative_22'] = exp['llm_proposed_decision']=='keep_in_conservative_safe_22'
exp['content_context_caveat'] = exp['llm_proposed_decision']=='candidate_for_expanded_with_caveat'
exp['plan'] = 'expanded_feature_set_candidate'
exp.to_csv(OUT_DIR / '05x_expanded_feature_set_candidate_contract.csv', index=False)
print(f'expanded candidate (incl conservative_22): {len(exp)} features')

# 3. forbidden/audit only
forbidden = decision_df[decision_df['llm_proposed_decision']=='forbidden_or_audit_only'].copy()
forbidden['removal_confirmed'] = 'no'
forbidden['user_approval_required_to_remove'] = forbidden['user_approval_required']
forbidden.to_csv(OUT_DIR / '05x_forbidden_or_audit_only_candidates.csv', index=False)
print(f'forbidden/audit only: {len(forbidden)} columns')

# 4. unresolved
unresolved = decision_df[decision_df['llm_proposed_decision']=='unresolved_user_review_required'].copy()
unresolved.to_csv(OUT_DIR / '05x_unresolved_user_review_required.csv', index=False)
print(f'unresolved: {len(unresolved)} columns')

# 5. user approval checklist
checklist = decision_df[decision_df['user_approval_required']=='yes'][[
    'column_name','feature_family','llm_proposed_decision','reason','caution','final_decision_status'
]].copy()
checklist['user_question'] = checklist.apply(lambda r:
    f'Do you approve including {r.column_name} in expanded_feature_set? Reason: {r.reason[:80]}'
    if r.llm_proposed_decision in ('candidate_for_expanded_feature_set','candidate_for_expanded_with_caveat')
    else f'Confirm handling of {r.column_name}: {r.reason[:80]}', axis=1)
checklist['user_decision'] = 'pending'
checklist.to_csv(OUT_DIR / '05x_user_approval_checklist.csv', index=False)
print(f'User approval checklist: {len(checklist)} items')

conservative_safe_22: 22 features
expanded candidate (incl conservative_22): 84 features
forbidden/audit only: 4 columns
unresolved: 1 columns
User approval checklist: 66 items


In [6]:
# SAFE/UNSAFE WORDING + NEXT STEP GATE

wording = pd.DataFrame([
    {'category':'feature_count','safe':'22개 conservative safe feature를 사용한 보수 baseline','unsafe':'91개 전체를 모두 넣었다'},
    {'category':'expanded_status','safe':'expanded_feature_set은 사용자 승인 전 후보 계약으로만 본다','unsafe':'확장 feature set을 이미 확정했다'},
    {'category':'leakage','safe':'end_date와 is_churn_prevented는 타이밍 확인 전 모델에 넣지 않는다','unsafe':'end_date나 is_churn_prevented를 feature로 사용했다'},
    {'category':'recency','safe':'recency의 reference date가 day21 이전임을 확인한 후 사용한다','unsafe':'recency를 타이밍 확인 없이 사용했다'},
    {'category':'genre_caveat','safe':'장르 비율은 day0-20 관측창 기준임을 확인한 후 사용한다','unsafe':'장르 비율 확인 없이 full feature로 투입했다'},
    {'category':'naming_caveat','safe':'total_watch_time은 실제로 day0-20 기준임을 09b로 확인했으나 명명 주의가 필요하다','unsafe':'total_watch_time이 전체 기간 데이터라고 설명했다'},
    {'category':'causality','safe':'프로모션 그룹 간 재구매율 차이는 관찰된 차이이다','unsafe':'100원딜이 이탈을 유발했다'},
    {'category':'analysis_unit','safe':'분석 단위는 row-level / subscription-event-level이다','unsafe':'unique user 분석이다'},
])
wording.to_csv(OUT_DIR / '05x_safe_unsafe_wording.csv', index=False)

gate = pd.DataFrame([
    {'condition':'user_approved_expanded_feature_set','required_before_06x':'yes','current_status':'not_yet',
     'note':'05x user_approval_checklist must be reviewed and approved before 06x dataset generation'},
    {'condition':'recency_timing_confirmed','required_before_06x':'yes','current_status':'not_yet',
     'note':'Confirm recency reference date is day21 or earlier'},
    {'condition':'end_date_is_churn_prevented_timing_confirmed','required_before_modeling':'yes','current_status':'not_yet',
     'note':'Confirm end_date and is_churn_prevented are historical/not post-scoring'},
    {'condition':'genre_observation_window_confirmed','required_before_modeling':'yes','current_status':'not_yet',
     'note':'Confirm genre ratios computed from day0-20 only'},
    {'condition':'no_modeling_before_approval','required_always':'yes','current_status':'enforced',
     'note':'Do not run 11/12/14/16/17 modeling steps before user approves feature contract'},
])
gate.to_csv(OUT_DIR / '05x_next_step_gate.csv', index=False)
print('Wording and gate files created')

Wording and gate files created


In [7]:
# FINAL CHECKS

def p(cond): return 'PASS' if cond else 'FAIL'

nb_path = NB_DIR / '05x_feature_contract_rebuild_260515.ipynb'
expected_csvs = [
    '05x_preflight_input_validation.csv',
    '05x_full_column_inventory.csv',
    '05x_feature_resolution_decision_table.csv',
    '05x_conservative_safe_22_contract.csv',
    '05x_expanded_feature_set_candidate_contract.csv',
    '05x_forbidden_or_audit_only_candidates.csv',
    '05x_unresolved_user_review_required.csv',
    '05x_user_approval_checklist.csv',
    '05x_safe_unsafe_wording.csv',
    '05x_next_step_gate.csv',
]

checks = pd.DataFrame([
    {'check':'all_outputs_inside_park_ingyeom','status':p(str(OUT_DIR).startswith(str(PARK))),'detail':str(OUT_DIR)},
    {'check':'raw_source_csv_not_modified','status':'PASS','detail':'read-only access only'},
    {'check':'notebook_exists','status':p(nb_path.exists()),'detail':str(nb_path)},
    {'check':'source_master_read','status':p(df_full is not None),'detail':f'{row_count} rows'},
    {'check':'all_91_columns_inventoried','status':p(len(inventory_df)==91),'detail':str(len(inventory_df))},
    {'check':'decision_table_created','status':p(len(decision_df)==91),'detail':str(len(decision_df))},
    {'check':'user_approval_required_column_exists','status':p('user_approval_required' in decision_df.columns),'detail':'yes'},
    {'check':'final_decision_status_pending','status':p((decision_df['final_decision_status']=='pending_user_approval').all()),'detail':'all pending'},
    {'check':'conservative_contract_created','status':p(len(c22)==22),'detail':str(len(c22))},
    {'check':'expanded_candidate_contract_created','status':p(len(exp)>22),'detail':str(len(exp))},
    {'check':'forbidden_audit_candidates_created','status':p(len(forbidden)>0),'detail':str(len(forbidden))},
    {'check':'unresolved_review_created','status':p(len(unresolved)>=0),'detail':str(len(unresolved))},
    {'check':'no_modeling_performed','status':'PASS','detail':'contract-only step'},
    {'check':'no_eda_performed','status':'PASS','detail':'contract-only step'},
    {'check':'no_shap_performed','status':'PASS','detail':'contract-only step'},
    {'check':'no_optuna_performed','status':'PASS','detail':'contract-only step'},
    {'check':'no_segmentation_performed','status':'PASS','detail':'contract-only step'},
    {'check':'all_expected_csvs_exist','status':p(all((OUT_DIR/f).exists() for f in expected_csvs)),'detail':str(expected_csvs)},
])

fail_count = int((checks['status'].str.startswith('FAIL')).sum())
checks = pd.concat([checks, pd.DataFrame([{
    'check':'critical_fail_count_zero','status':p(fail_count==0),'detail':str(fail_count)
}])], ignore_index=True)

checks.to_csv(OUT_DIR / '05x_final_checks.csv', index=False)
print(checks.to_string(index=False))
print(f'\nFAIL count: {fail_count}')

                               check status                                                                                                                                                                                                                                                                                                                                                                                                detail
     all_outputs_inside_park_ingyeom   PASS                                                                                                                                                                                                                                                                                                          C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\05x_feature_contract_rebuild_260515
         raw_source_csv_not_modified   PASS                                                                                         

In [8]:
# README
readme = '''# 05x_feature_contract_rebuild_260515

## 목적
기존 05~14 pre-13b 산출물이 archive로 격리된 상태에서, 91개 전체 컬럼을 재검토하고
보수 플랜과 확장 플랜의 feature contract를 새로 작성한다.

## 왜 05x부터 다시 시작하는가
기존 pre-13b 작업은 conservative_safe_22 기준으로만 진행되어, 91개 컬럼 중 장르/recency/
membership 등 69개 review 컬럼이 "나중에" 상태로 방치되었다.
이번 05x는 91개 전체를 명시적으로 재검토하고 사용자 승인 체계를 만든다.

## 기존 pre-13b 22개 결과의 지위
archive/pre13b_conservative_safe_22_reference에 보존된 pre-13b 결과는 reference/deprecated이다.
canonical 기준선으로 복원하지 않는다. 단, conservative_safe_22 22개 feature 목록은 이번
05x에서도 보수 baseline 기준선으로 유지한다.

## LLM 최종 피처 결정 원칙
LLM(Codex/Claude)은 feature를 최종 제외하거나 승격하지 않는다.
LLM은 근거와 후보만 제시하며, 최종 사용 여부는 반드시 사용자 승인 후 확정한다.

## 보수 플랜 (conservative_safe_22)
- 기존 22개 weekly-window safe feature
- pre-13b 결과와 비교 가능한 기준선
- 추가 검증 없이 즉시 모델 투입 가능

## 확장 플랜 (expanded_feature_set)
- membership/context, total usage aggregate, content/genre 포함 후보
- 사용자 승인 전 후보 상태
- context/content caveat flag 포함

## 사용자 승인 전 다음 단계 금지
05x_user_approval_checklist.csv 검토 및 승인 전에 06x로 진행하지 않는다.
11/12/14/16/17 모델링 단계 진입 금지 유지.

## 다음 단계
사용자 승인 후 → 06x_dataset_generation
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('README created')

README created


In [9]:
# NOTE.MD UPDATE
note_entry = f'''
## {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | 05x_feature_contract_rebuild_260515

- purpose: 기존 05~14 pre-13b 산출물이 archive로 격리된 상태에서 91개 전체 컬럼을 재검토하고 사용자 승인용 feature contract를 작성했다.
- pre-13b 지위: 05~14 pre-13b 결과는 _archive/pre13b_conservative_safe_22_reference에 보존됨. canonical 복원 안 함.
- conservative_safe_22 count: {len(c22)}
- expanded_feature_set candidate count (incl conservative_22): {len(exp)}
- forbidden_or_audit_only count: {len(forbidden)}
- unresolved count: {len(unresolved)}
- user_approval_checklist items: {len(checklist)}
- LLM 원칙: LLM은 feature 최종 제외/승격을 결정하지 않는다. 근거와 후보만 제시. 최종 결정은 사용자 승인 후 확정.
- final_checks: {'PASS' if fail_count == 0 else 'FAIL'} (fail_count={fail_count})
- output_dir: {OUT_DIR}
- next step: 06x_dataset_generation (사용자 승인 후 진행)
- gate: 05x_user_approval_checklist.csv 승인 전 06x/11/12/14/16/17 진행 금지
'''

with open(NOTE_MD, 'a', encoding='utf-8') as f:
    f.write(note_entry)
print(f'note.md updated: {NOTE_MD}')

note.md updated: C:\Code\ott-churn-prediction\park.ingyeom\note.md


In [10]:
# ZIP CREATION AND VERIFICATION
files_to_zip = list(OUT_DIR.glob('*.csv')) + list(OUT_DIR.glob('*.md'))
files_to_zip.append(NB_DIR / '05x_feature_contract_rebuild_260515.ipynb')

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_zip:
        if f.exists():
            zf.write(f, f.name)

with zipfile.ZipFile(ZIP_OUT, 'r') as zf:
    names = zf.namelist()

print(f'ZIP created: {ZIP_OUT}')
print(f'ZIP contents ({len(names)} files):')
for n in sorted(names):
    print(f'  {n}')

ZIP created: C:\Code\ott-churn-prediction\park.ingyeom\zip\05x_feature_contract_rebuild_260515_review_package.zip
ZIP contents (13 files):
  05x_conservative_safe_22_contract.csv
  05x_expanded_feature_set_candidate_contract.csv
  05x_feature_contract_rebuild_260515.ipynb
  05x_feature_resolution_decision_table.csv
  05x_final_checks.csv
  05x_forbidden_or_audit_only_candidates.csv
  05x_full_column_inventory.csv
  05x_next_step_gate.csv
  05x_preflight_input_validation.csv
  05x_safe_unsafe_wording.csv
  05x_unresolved_user_review_required.csv
  05x_user_approval_checklist.csv
  README.md
